In [1]:
# Import relevant packages
import gamspy as gp
# from gamspy import Container
# from gams import transfer
# import gams.transfer as gt
# import gdxpds
import numpy as np
import pandas as pd
from pathlib import Path
import re
import os

In [2]:
os.getcwd()

'/Users/ahanamukherjee/seer/seer/gams2pypsa'

### Data Pre-Processing & Exploration

In [3]:
# Define a function to load all .gdx files 
def load_gdx_files_simple(data_dir): 

    # Initialize an empty dictionary to store the data 
    all_data = {}
    for country_dir in Path(data_dir).iterdir(): 
        gdx_file = country_dir / 'input.gdx' 
        if gdx_file.exists():
            m = gp.Container() 
            m.read(str(gdx_file)) 
            # Convert all symbols into dataframes 
            country_data = {}
            for symbol_name in m.listSymbols(): 
                country_data[symbol_name] = pd.DataFrame(m[symbol_name].records) 

            all_data[country_dir.name] = country_data 
    
    return all_data

In [4]:
# Load all of the gdx files into a dictionary
all_data_simple = load_gdx_files_simple("data/raw")

In [5]:
# Check if all of the symbol names are the same across all countries
country_symbols = {country: set(symbols.keys()) for country, symbols in all_data_simple.items()}
symbol_check = all(s == list(country_symbols.values())[0] for s in country_symbols.values())
print(f'Does all country data use the same set of symbols? {symbol_check}')
print("")

# Find the "common" symbols across all countries
common_symbols = set.intersection(*country_symbols.values())
print("Common symbols in all countries:")
print(common_symbols)
print("")

# Find country-specific differences
for country, symbols in country_symbols.items():
    unique_to_country = symbols - common_symbols
    if unique_to_country:
        print(f"Symbols unique to {country}: {unique_to_country}")

Does all country data use the same set of symbols? False

Common symbols in all countries:
{'pFuelPrice', 'pDemandProfile', 'pHours', 'pVREgenProfile', 'pDemandForecast', 'pNewTransmission', 'pEnergyEfficiencyFactor', 'pCarbonPrice', 'ftfindex', 'pEmissionsTotal', 'pVREProfile', 'pAvailability', 'pTradePrice', 'pDemandData', 'pStorDataExcel', 'pTransferLimit', 'pCSPData', 'pMaxFuellimit', 'y'}

Symbols unique to Eswatini: {'pH2DataExcel', 'pFuelData', 'pGenDataExcel', 'pPlanningReserveMargin', 'pSpinningReserveReqSystem', 'Relevant', 'pScalars', 'peak', 'pSpinningReserveReqCountry', 'pEmissionsCountry', 'pExtTransferLimit', 'pCapexTrajectory', 'pTechDataExcel', 'zext', 'pZoneIndex', 'sTopology', 'hh', 'MapGG', 'pLossFactor', 'pMaxExchangeShareTotal', 'zcmapExcel', 'pMaxExchangeShare', 'pAvailabilityH2', 'pFuelTypeCarbonContent', 'pCapexTrajectoryH2', 'pExternalH2'}
Symbols unique to Tanzania: {'pAnnualMaxBuild', 'pH2DataExcel', 'pFuelData', 'pGenDataExcel', 'pPlanningReserveMargin', 'p

In [6]:
for symbol in common_symbols:
    print(f"\nChecking symbol: {symbol}")
    
    base_country = list(all_data_simple.keys())[0]
    base_cols = set(all_data_simple[base_country][symbol].columns)
    
    for country in all_data_simple:
        cols = set(all_data_simple[country][symbol].columns)
        
        if cols != base_cols:
            print(f"  Differences in {country}:")
            print(f"    Missing: {base_cols - cols}")
            print(f"    Extra:   {cols - base_cols}")


Checking symbol: pFuelPrice
  Differences in EAPP:
    Missing: {'uni_0', 'uni_1', 'uni_2'}
    Extra:   {'c', 'f', 'y'}
  Differences in Gambia:
    Missing: {'uni_0', 'uni_1', 'uni_2'}
    Extra:   {'c', 'f', 'y'}
  Differences in CAPP:
    Missing: {'uni_0', 'uni_1', 'uni_2'}
    Extra:   {'fuel', 'uni', 'zone'}
  Differences in Rwanda:
    Missing: {'uni_0', 'uni_1', 'uni_2'}
    Extra:   {'c', 'f', 'y'}

Checking symbol: pDemandProfile
  Differences in EAPP:
    Missing: {'uni_0', 'uni_1', 'uni_3', 'uni_2'}
    Extra:   {'t', 'q', 'd', 'z'}
  Differences in Gambia:
    Missing: {'uni_0', 'uni_1', 'uni_3', 'uni_2'}
    Extra:   {'t', 'q', 'd', 'z'}
  Differences in Guinea:
    Missing: {'uni_0', 'uni_1', 'uni_3', 'uni_2'}
    Extra:   {'ZONE', 'daytype', 'uni', 'Season'}
  Differences in CAPP:
    Missing: {'uni_0', 'uni_1', 'uni_3', 'uni_2'}
    Extra:   {'daytype', 'uni', 'season', 'zone'}
  Differences in Rwanda:
    Missing: {'uni_0', 'uni_1', 'uni_3', 'uni_2'}
    Extra:   {'

In [7]:
# Obtain all of the symbols used across dataset
all_symbols = list(set().union(*country_symbols.values()))
all_symbols.sort()

# Convert to a Pandas series and output to a CSV
all_symbols = pd.Series(all_symbols)
all_symbols.to_csv('data/processed/all_symbols.csv', index=False)

In [8]:
def audit_fuel_tech_values(all_data: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Extract every unique fuel-type string (from ftfindex) and every unique
    technology value (from pTechDataExcel) seen across all countries.

    Returns
    -------
    fuel_summary : pd.DataFrame
        Columns: fuel_id, fuel_name, countries
        One row per unique (fuel_id, fuel_name) pair, with a list of countries
        that carry it.

    tech_summary : pd.DataFrame
        Columns: tech_id, tech_name, countries
        One row per unique (tech_id, tech_name) pair from pTechDataExcel.
    """

    fuel_records = []   # (fuel_id, fuel_name, country)
    tech_records = []   # (tech_id, tech_name, country)

    for region, country_data in all_data.items():

        # ── ftfindex ─────────────────────────────────────────────────────────
        ftf = country_data.get("ftfindex")
        if ftf is not None and not ftf.empty:
            ftf = ftf.copy()
            ftf.columns = ftf.columns.str.strip().str.lower()

            name_col = None  # not present in all countries

            if len(ftf.columns) >= 3:
                type_col = ftf.columns[0]   # General fuel type
                name_col = ftf.columns[1]   # Fuel type name (country specific)
                id_col = ftf.columns[2]     # Fuel id
            else:
                type_col = ftf.columns[0]   # General fuel type
                id_col = ftf.columns[1]     # Fuel id
            
            for _, row in ftf.iterrows():
                fuel_records.append({
                    "fuel_id": row[id_col],
                    "fuel_type": str(row[type_col]).strip(),
                    "fuel_name": re.sub(r"^\d+\.", "", str(row[name_col]).strip()) if name_col else None,
                    "country": region,   
                })

        # ── pTechDataExcel ───────────────────────────────────────────────────
        tech = country_data.get("pTechDataExcel")
        if tech is not None and not tech.empty:
            tech = tech.copy()
            tech.columns = tech.columns.str.strip().str.lower()

            name_col = tech.columns[0]      # Technology name
            id_col = tech.columns[2]        # Technology value id
            # param_col = None
            # val_col = "value" if "value" in tech.columns else tech.columns[-1]

            for _, row in tech.iterrows():
                tech_records.append({
                    "tech_id":   row[id_col],
                    "tech_name": str(row[name_col]).strip(),
                    "country":   region,
                })

    # ── Summarise fuels ──────────────────────────────────────────────────────
    if fuel_records:
        fuel_df = pd.DataFrame(fuel_records)
        fuel_summary = (
            fuel_df.groupby(["fuel_type", "fuel_name"], sort=True)["country"]
            .apply(lambda x: ", ".join(sorted(set(x))))
            .reset_index()
            .rename(columns={"country": "countries"})
            .sort_values("fuel_type")
            .reset_index(drop=True)
        )
    else:
        fuel_summary = pd.DataFrame(columns=["fuel_id", "fuel_type", "fuel_name", "countries"])

    # ── Summarise technologies ───────────────────────────────────────────────
    if tech_records:
        tech_df = pd.DataFrame(tech_records)
        tech_summary = (
            tech_df.groupby(["tech_name"], sort=True)["country"]
            .apply(lambda x: ", ".join(sorted(set(x))))
            .reset_index()
            .rename(columns={"country": "countries"})
            .sort_values("tech_name")
            .reset_index(drop=True)
        )
    else:
        tech_summary = pd.DataFrame(columns=["tech_id", "tech_name", "countries"])

    return fuel_summary, tech_summary

# Run the function 
fuel_summary, tech_summary = audit_fuel_tech_values(all_data_simple)

print(f"Unique fuel types: {len(fuel_summary)}")
display(fuel_summary)

print(f"\nUnique technology entries: {len(tech_summary)}")
display(tech_summary)

# Export both summaries for reference when building FUEL_TO_TECH
fuel_summary.to_csv("data/references/audit_fuel_types.csv", index=False)
tech_summary.to_csv("data/references/audit_tech_types.csv", index=False)

Unique fuel types: 61


,fuel_type,fuel_name,countries
0,Battery,BESS,Serbia
1,Battery,BESS1h,WAPP
2,Battery,BESS2h,WAPP
3,Battery,BESS3h,WAPP
4,Battery,Battery,"Eswatini, Guinea, Malawi, Tanzania, Timor Lest..."
...,...,...,...
56,Wind,OffshoreWind,"Malawi, Tanzania, Turkiye"
57,Wind,OnshoreWind,"Malawi, Tanzania, Turkiye"
58,Wind,Wind,"Eswatini, Guinea, Timor Leste, WAPP"
59,Wind,WindEx,Serbia



Unique technology entries: 31


,tech_name,countries
0,BIOGAS,"Eswatini, Guinea, Malawi, Serbia, Tanzania, Ti..."
1,BIOMAS,"Eswatini, Malawi, Serbia, Tanzania, Timor Lest..."
2,Biomas,"Guinea, WAPP"
3,CCGT,"Eswatini, Guinea, Malawi, Serbia, Tanzania, Ti..."
4,CCGTH2,"Eswatini, Serbia, Tanzania, Timor Leste, Turkiye"
5,CHP,"Eswatini, Guinea, Malawi, Serbia, Tanzania, Ti..."
6,COAL,Turkiye
7,CSP,"Eswatini, Guinea, Malawi, Serbia, Tanzania, Ti..."
8,Coal,"Eswatini, Guinea, Malawi, Serbia, Tanzania, Ti..."
9,CoalCCS,"Eswatini, Serbia, Tanzania, Timor Leste, Turkiye"


### Input File Processing

In [9]:
# Define a function to build the symbol mapping schema
def build_symbol_registry(mapping_path):

    # Read in the CSV file for symbol mapping
    mapping_df = pd.read_csv(mapping_path)

    # Eliminate any whitespaces in names
    for col in mapping_df.select_dtypes(include='object'):
        mapping_df[col] = mapping_df[col].str.strip()

    # Extract the column names for each symbol
    col_names = [c for c in mapping_df.columns if c.startswith('col_')]

    # Initilize empty dictionary to store symbol mapping
    symbol_registry = {}

    # Build the dictionary
    for _, row in mapping_df.iterrows():

        # Define the raw and standardized symbol name
        resolved = row['canonical']
        raw_name = row['raw']

        # Define the cols
        cols = [row[c] for c in col_names if pd.notna(row[c]) and row[c] != ""]

        # Initialize a new canonical symbol if it doesn't exist in dictionary
        if resolved not in symbol_registry:
            symbol_registry[resolved] = {
                'raw_names': [],
                'columns': cols
            }

        # Append raw name aliases to the canonical symbol
        symbol_registry[resolved]['raw_names'].append(raw_name)

    return symbol_registry


# Define a function to look up the raw symbol name in the symbol registry
def symbol_lookup(raw_symbol, registry):

    for resolved_name, dict_items in registry.items():
        if raw_symbol in dict_items['raw_names']:
            return resolved_name
        
    return None

In [10]:
# # Define a function infer column names for symbol data
# def infer_names(col_series):
#     """Attempt to infer standardized column name from column value types"""

#     # Extract a sample of data from the column for text analysis
#     sample = col_series.dropna().astype(str).head(50)

#     if sample.empty:
#         return None
    
#     # Hour (t01, t1, t24, etc)
#     if sample.str.match(r"^t\d+$", case=False).all():
#         return "hour"
    
#     # Year detection (4-digit years)
#     if sample.str.match(r"^\d{4}$").all():
#         return "year"

#     # Season detection (dry/wet, Q1, m1, etc.)
#     if sample.str.match(r"^(dry|wet|Q\d+|m\d+)$", case=False).all():
#         return "season"

#     # Data detecton (d1, d2, d3)
#     if sample.str.match(r"^d\d+$", case=False).all():
#         return "day"

#     # Day detection (small integers)
#     if sample.str.match(r"^\d+$").all():
#         max_val = sample.astype(int).max()
#         if max_val <= 31:
#             return "day"
    
#     # Parameter column detection
#     # Heuristic: mostly alphabetic labels
#     if sample.str.match(r"[A-Za-z]+[A-Za-z0-9]"):
#         return "parameters"

#     return None

In [11]:
def symbol_sync(df, symbol_obj, resolved_name, registry, strict=False):
    """
    Map symbol column names to defined naming schema.
    Align dataframe columns to canonical maximum schema using
    ordered subsequence + left-right anchoring logic.

    Special handling:
        - element_text is protected and mapped explicitly
        - No other column may map to element_text
    """

    if resolved_name not in registry:
        return df

    # Obtain the maximum expected number of columns for the symbol
    expected_cols = registry[resolved_name].get("columns", [])
    if not expected_cols:
        return df

    # If symbol does not have any columns or symbol df is empty, return the original df
    if df.empty or len(df.columns) == 0:
        return pd.DataFrame(columns=expected_cols)

    # Create a copy of the symbol dataframe for column re-naming and processing
    df = df.copy()

    # Check that "value" column exists or rename last column to value
    if "value" not in df.columns:
        df = df.rename(columns={df.columns[-1]: "value"})

    # Extract all expected column names from schema excluding "value" column
    expected_dims = [c for c in expected_cols if c != "value"]

    # Extract all actual column names from df excluding "value" column
    actual_dim_cols = [c for c in df.columns if c != "value"]

    # Explicitly map element_text column if present in dataframe
    rename_map = {}

    # Remove element_text from canonical columns for dataframe
    if "element_text" in expected_dims and "element_text" in df.columns:
        rename_map["element_text"] = "element_text"
        actual_dim_cols.remove("element_text")

    # Determine the remaining dimensions that need to be mapped
    remaining_dims = [
        d for d in expected_dims if d != "element_text"
    ]

    # Use a left-to-right anchoring scheme for renaming columns
    # Determine the total number of columns that need to be mapped
    n = len(actual_dim_cols)
    # n = len(dims_for_alignment)

    def left_to_right_mapping(expected_dims, n):
        if n == 0:
            return []

        if n >= len(expected_dims):
            return expected_dims[:n]

        if n == 1:
            return [expected_dims[0]]

        left_anchor = expected_dims[0]
        right_anchor = expected_dims[-1]

        remaining = n - 2
        middle = expected_dims[1:-1]

        return (
            [left_anchor] +
            middle[:remaining] +
            [right_anchor]
        )

    # Apply mapping for remaining columns based on column order defined in schema
    mapped_dims = left_to_right_mapping(remaining_dims, n)

    # Assign renamed dimensions
    for i in range(len(mapped_dims)):
        rename_map[actual_dim_cols[i]] = mapped_dims[i]
    # Rename columns according to mapped dimensions
    df = df.rename(columns=rename_map)

    # Add extra columns that are not in the actual df but in expected column schema as NaNs
    for col in expected_dims:
        if col not in df.columns:
            df[col] = np.nan

    # If "value" column doens't exist in the dataframe, add column with NaNs
    if "value" not in df.columns:
        df["value"] = np.nan

    # Reorder columns to match the expected column schema
    df = df[expected_cols]

    # Raise ValueError if mismatch across expected schema and actual columns
    if strict:
        if list(df.columns) != expected_cols:
            raise ValueError(
                f"{resolved_name}: schema mismatch after alignment"
            )

    return df

In [12]:
# Define a function to load all .gdx files 
def load_gdx_files(data_dir, mapping_path, strict): 

    # Build the symbol mapping registry
    registry = build_symbol_registry(mapping_path)

    # Initialize an empty dictionary to store the data 
    all_data = {}
    for country_dir in Path(data_dir).iterdir(): 
        gdx_file = country_dir / 'input.gdx' 
        if gdx_file.exists():

            print(f'Processing {country_dir.name}')

            m = gp.Container() 
            m.read(str(gdx_file)) 

            # Convert all symbols into dataframes 
            country_data = {}

            for raw_symbol in m.listSymbols(): 

                # Find the corresponding resolved symbol name in the symbol registry
                resolved_symbol = symbol_lookup(raw_symbol, registry)

                if resolved_symbol is None:
                    continue
                
                # Convert the symbol records into a dataframe
                symbol_obj = m[raw_symbol]
                df = pd.DataFrame(symbol_obj.records)

                # Rename columns of the symbol data
                df = symbol_sync(df, symbol_obj, resolved_symbol, registry, strict)
                
                # Append cleaned df to country
                country_data[resolved_symbol] = df

            all_data[country_dir.name] = country_data 
    
    return all_data

In [13]:
# Load all of the gdx files into a dictionary
all_data = load_gdx_files(data_dir="data/raw", 
                          mapping_path="data/references/symbol_mapping.csv", 
                          strict=True)

Processing Eswatini
Processing Tanzania


/var/folders/x0/qnl8p2f174g_y6029kr9p3q80000gn/T/ipykernel_11277/2562891664.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in mapping_df.select_dtypes(include='object'):


Processing EAPP
Processing Malawi
Processing WAPP
Processing Serbia
Processing Timor Leste
Processing Gambia
Processing Guinea
Processing CAPP
Processing Rwanda
Processing Turkiye


#### Generator Additional Data Processing

In [14]:
# Define fuel, technology pair mapping based on fuel and technology types across acountries
# Keys are the stripped fuel_name strings from ftfindex (e.g. "CCGTEx").
# Values are (fuel_label, tech_label) for populating pGenDataInputCustom.

FUEL_NAME_MAP = {
    # ── Battery ──────────────────────────────────────────────────────────────
    "BESS":             ("Battery",     "STORAGE"),
    "BESS1h":           ("Battery",     "STORAGE"),
    "BESS2h":           ("Battery",     "STORAGE"),
    "BESS3h":           ("Battery",     "STORAGE"),
    "Battery":          ("Battery",     "STORAGE"),

    # ── Biomass ───────────────────────────────────────────────────────────────
    "Woodchip":         ("Biomass",     "BiomassPlant"),
    "Biomass":          ("Biomass",     "BiomassPlant"),
    "Bagasse":          ("Biomass",     "BiomassPlant"),
    "Biomas":           ("Biomass",     "BiomassPlant"),   # note: case differs in tech audit
    "Biomass_CCS":      ("Biomass",     "BiomassPlant"),

    # ── Coal ─────────────────────────────────────────────────────────────────
    "Coal":             ("Coal",        "ICE"),
    "CoalEx":           ("Coal",        "ICE"),
    "CoalNew":          ("Coal",        "ICE"),
    "DomesticCoal":     ("Coal",        "ICE"),
    "ImportedCoal":     ("Coal",        "ICE"), 

    # ── Coal CCS ─────────────────────────────────────────────────────────────
    "CoalCCS":          ("CoalCCS",     "ICE"),
    "Coal_CCS":         ("CoalCCS",     "ICE"),

    # ── Custom / Nuclear / Geothermal ────────────────────────────────────────
    "Uranium":          ("CustomFuel",  "NUC"),
    "Nuclear":          ("CustomFuel",  "NUC"),
    "Geothermal":       ("CustomFuel",  "GEO"),      # also appears under Geothermal fuel_type

    # ── Diesel ───────────────────────────────────────────────────────────────
    "Diesel":           ("Diesel",      "ICE"),

    # ── Gas ──────────────────────────────────────────────────────────────────
    "CCGTEx":           ("Gas",         "CCGT"),
    "CCGTNew":          ("Gas",         "CCGT"),
    "OCGTEx":           ("Gas",         "OCGT"),
    "OCGTNew":          ("Gas",         "OCGT"),
    "Gas":              ("Gas",         "CCGT"),     # verify: could also be GT depending on country
    "LNG":              ("Gas",         "CCGT"),

    # ── Gas CCS ──────────────────────────────────────────────────────────────
    "CCGT_CCS":         ("GasCCS",      "CCGT"),
    "GasCCS":           ("GasCCS",      "GasCCS"),

    # ── Geothermal ───────────────────────────────────────────────────────────
    "Geothermal":       ("Geothermal",  "GEO"),      # fuel_type=Geothermal (Tanzania, Turkiye)

    # ── HFO ──────────────────────────────────────────────────────────────────
    "HFO":              ("HFO",         "ICE"),
    "FuelOil":          ("HFO",         "ICE"),

    # ── Hydro ────────────────────────────────────────────────────────────────
    "Hydro":            ("Hydro",       "STO HY"),   # verify: generic — could be ROR or reservoir
    "HydroStorEx":      ("Hydro",       "STO HY"),
    "HydroStorNew":     ("Hydro",       "STO HY"),
    "MPA_Main":         ("Hydro",       "ReservoirHydro"),
    "MPA_Opt":          ("Hydro",       "ReservoirHydro"),
    "MPA_Reg":          ("Hydro",       "ReservoirHydro"),
    "PHP":              ("Hydro",       "ReservoirHydro"),
    "RoREx":            ("Hydro",       "ROR"),
    "RoRNew":           ("Hydro",       "ROR"),
    "Water":            ("Hydro",       "STO HY"),   # verify: generic — confirm ROR vs reservoir
    "PumpedHy":         ("Hydro",       "STO HY"),

    # ── Hydrogen ─────────────────────────────────────────────────────────────
    "GreenHydrogen":    ("Hydrogen",    "CCGTH2"),
    "Hydrogen":         ("Hydrogen",    "CCGTH2"),

    # ── Import ───────────────────────────────────────────────────────────────
    "Imports":          ("Import",      "ImportTransmission"),
    "Import_SAR":       ("Import",      "ImportTransmission"),
    "Import_SAPP":      ("Import",      "ImportTransmission"),
    "Import_EDM":       ("Import",      "ImportTransmission"),
    "Import":           ("Import",      "ImportTransmission"),

    # ── Solar ────────────────────────────────────────────────────────────────
    "CSP":              ("Solar",       "CSPPlant"),
    "PV":               ("Solar",       "PV"),
    "PVEx":             ("Solar",       "PV"),
    "PVNew":            ("Solar",       "PV"),
    "Rooftop":          ("Solar",       "PV"),       # verify: could be PVwSTO if co-located storage

    # ── Wind ─────────────────────────────────────────────────────────────────
    "Offshore":         ("Wind",        "OffshoreWind"),
    "OffshoreWind":     ("Wind",        "OffshoreWind"),
    "OnshoreWind":      ("Wind",        "OnshoreWind"),
    "Wind":             ("Wind",        "Wind"),     # verify: onshore vs offshore
    "WindEx":           ("Wind",        "Wind"),
    "WindNew":          ("Wind",        "Wind"),
}

In [15]:
# Build per-generator lookup from ftfindex
# For each country, maps the numeric fuel_id in pGenDataExcel to (fuel_label, tech_label) via FUEL_NAME_MAP

def build_fuel_id_lookup(ftf: pd.DataFrame) -> dict:
    """
    Build a {fuel_id: (fuel_label, tech_label)} dict from a country's ftfindex.
    Returns an empty dict if the structure is not recognised.
    """
    ftf = ftf.copy()
    ftf.columns = ftf.columns.str.strip().str.lower()

    if len(ftf.columns) < 2:
        return {}

    if len(ftf.columns) >= 3:
        name_col = ftf.columns[1]
        id_col   = ftf.columns[2]
    else:
        name_col = None
        id_col   = ftf.columns[1]

    lookup = {}
    for _, row in ftf.iterrows():
        fuel_id   = row[id_col]
        fuel_name = re.sub(r"^\d+\.", "", str(row[name_col]).strip()) if name_col else None
        mapped    = FUEL_NAME_MAP.get(fuel_name)
        if mapped:
            lookup[fuel_id] = mapped
        else:
            if fuel_name:
                print(f"  [warn] no mapping found for fuel_name: '{fuel_name}'")

    return lookup

In [16]:
# populate zone, tech, fuel for pGenDataInputCustom
def enrich_gen_data(region: str, country_data: dict, country_data_simple: dict) -> pd.DataFrame | None:
    """
    Populate missing zone, fuel, and tech columns in pGenDataInputCustom
    using pZoneIndex, ftfindex, and FUEL_NAME_MAP.

    Parameters
    ----------
    region : str
        Country/region name for logging.
    country_data : dict
        Processed symbol dict from all_data — provides pGenDataInputCustom.
    country_data_simple : dict
        Raw symbol dict from all_data_simple — provides pGenDataExcel,
        pZoneIndex, and ftfindex.
    """

    # pGenDataInputCustom from processed data
    if "pGenDataInputCustom" not in country_data or country_data["pGenDataInputCustom"] is None or country_data["pGenDataInputCustom"].empty:
        print(f"  [{region}] skipping — missing pGenDataInputCustom")
        return None

    # Lookup symbols from raw data
    for sym in ["pGenDataExcel", "pZoneIndex", "ftfindex"]:
        if sym not in country_data_simple or country_data_simple[sym] is None or country_data_simple[sym].empty:
            print(f"  [{region}] skipping — missing {sym} in all_data_simple")
            return None

    target  = country_data["pGenDataInputCustom"].copy()
    gen_raw = country_data_simple["pGenDataExcel"].copy()
    zones   = country_data_simple["pZoneIndex"].copy()
    ftf     = country_data_simple["ftfindex"].copy()

    # Normalise column names
    target.columns  = target.columns.str.strip().str.lower()
    gen_raw.columns = gen_raw.columns.str.strip().str.lower()
    zones.columns   = zones.columns.str.strip().str.lower()

    # ── Zone mapping ─────────────────────────────────────────────────────────
    zone_id_col   = next((c for c in zones.columns if pd.to_numeric(zones[c], errors="coerce").notna().all()), None)
    zone_name_col = next((c for c in zones.columns if c != zone_id_col), None)

    if zone_id_col and zone_name_col:
        zone_map = (
            zones.assign(**{zone_id_col: pd.to_numeric(zones[zone_id_col], errors="coerce")})
            .dropna(subset=[zone_id_col])
            .set_index(zone_id_col)[zone_name_col]
            .to_dict()
        )
    else:
        zone_map = {}
        print(f"  [{region}] warn — could not build zone map")

    # ── Pivot pGenDataExcel to wide format ───────────────────────────────────
    # Identify the parameter column (contains "zone", "fuel1", etc.)
    param_col = next(
        (c for c in gen_raw.columns if c != "value"
         and gen_raw[c].dropna().astype(str).str.lower().isin(["zone", "fuel1", "fuel2"]).any()),
        None
    )
    gen_col = next((c for c in gen_raw.columns if c not in (param_col, "value")), None)

    if not param_col or not gen_col:
        print(f"  [{region}] warn — could not identify columns in pGenDataExcel")
        return target

    gen_wide = (
        gen_raw[[gen_col, param_col, "value"]]
        .dropna(subset=[param_col])
        .pivot_table(index=gen_col, columns=param_col, values="value", aggfunc="first")
        .reset_index()
    )
    gen_wide.columns = [str(c).strip().lower() for c in gen_wide.columns]
    gen_wide = gen_wide.rename(columns={gen_col.lower(): "generator"})

    # ── Build fuel_id → (fuel_label, tech_label) lookup from ftfindex ────────
    fuel_id_lookup = build_fuel_id_lookup(ftf)

    # ── Resolve zone, fuel, tech per generator ───────────────────────────────
    if "zone" in gen_wide.columns:
        gen_wide["_zone"] = (
            pd.to_numeric(gen_wide["zone"], errors="coerce").map(zone_map)
        )
    else:
        gen_wide["_zone"] = np.nan

    # Prefer fuel1, fall back to fuel2
    fuel_id_series = None
    if "fuel1" in gen_wide.columns:
        fuel_id_series = pd.to_numeric(gen_wide["fuel1"], errors="coerce")
    if "fuel2" in gen_wide.columns:
        fuel2 = pd.to_numeric(gen_wide["fuel2"], errors="coerce")
        fuel_id_series = fuel_id_series.fillna(fuel2) if fuel_id_series is not None else fuel2

    if fuel_id_series is not None and fuel_id_lookup:
        gen_wide["_fuel"] = fuel_id_series.map({k: v[0] for k, v in fuel_id_lookup.items()})
        gen_wide["_tech"] = fuel_id_series.map({k: v[1] for k, v in fuel_id_lookup.items()})
    else:
        gen_wide["_fuel"] = np.nan
        gen_wide["_tech"] = np.nan
        print(f"  [{region}] warn — fuel/tech mapping could not be applied")

    # ── Identify generator id column in target ───────────────────────────────
    skip_cols = {"zone", "fuel", "tech", "technology", "value", "parameters"}
    gen_id_col = next((c for c in target.columns if c not in skip_cols), None)

    if not gen_id_col:
        print(f"  [{region}] warn — could not identify generator id column in pGenDataInputCustom")
        return target

    # ── Merge enrichment into target (fill NaNs only) ────────────────────────
    enrichment = gen_wide[["generator", "_zone", "_fuel", "_tech"]].drop_duplicates("generator")
    target = target.merge(enrichment, how="left", left_on=gen_id_col, right_on="generator")

    for target_col, enrich_col in [("zone", "_zone"), ("fuel", "_fuel"), ("tech", "_tech")]:
        if target_col in target.columns and enrich_col in target.columns:
            target[target_col] = target[target_col].fillna(target[enrich_col])

    target = target.drop(columns=["generator", "_zone", "_fuel", "_tech"], errors="ignore")

    return target

In [17]:
# Enrich all region/country generator-specific dataframes
def enrich_all_gen_data(all_data: dict, all_data_simple: dict) -> dict:
    """
    Apply enrich_gen_data() to every country/region.

    Parameters
    ----------
    all_data : dict
        Processed data dict — source of pGenDataInputCustom.
    all_data_simple : dict
        Raw data dict — source of pGenDataExcel, pZoneIndex, ftfindex.
    """
    enriched = {}
    for region, country_data in all_data.items():
        print(f"Enriching: {region}")

        # Skip if region is absent from raw data
        if region not in all_data_simple:
            print(f"  [{region}] skipping — not found in all_data_simple")
            enriched[region] = country_data
            continue

        updated = dict(country_data)
        result = enrich_gen_data(region, country_data, all_data_simple[region])
        if result is not None:
            updated["pGenDataInputCustom"] = result
        enriched[region] = updated

    return enriched

In [18]:
# Enrich all generator-specific data
all_data_enriched = enrich_all_gen_data(all_data, all_data_simple)

Enriching: Eswatini
Enriching: Tanzania
Enriching: EAPP
  [EAPP] skipping — missing pZoneIndex in all_data_simple
Enriching: Malawi
Enriching: WAPP
Enriching: Serbia
Enriching: Timor Leste
Enriching: Gambia
  [Gambia] skipping — missing pZoneIndex in all_data_simple
Enriching: Guinea
Enriching: CAPP
  [CAPP] skipping — missing pGenDataExcel in all_data_simple
Enriching: Rwanda
  [Rwanda] skipping — missing pZoneIndex in all_data_simple
Enriching: Turkiye


#### Write out processed data to CSVs

In [19]:
# Define a function to write out all model data into csvs
def output_to_csv(all_data, output_dir):

    # Create the output directory if it does not exist
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Loop through each country/region and generate csv for each dataframe
    for region_name, region_data in all_data.items():

        # Create region-specific directory
        region_dir = output_dir / region_name
        region_dir.mkdir(parents=True, exist_ok=True)

        # Loop through the symbol dataframes for each region
        for symbol_name, df in region_data.items():

            # Skip writing empty dataframes
            if df is None or df.empty:
                continue

            # Define the file path and write out the dataframe to a csv
            file_path = region_dir / f'{symbol_name}.csv'
            df.to_csv(file_path, index=False)
        
        # Print logging message
        print(f"Finished writing all symbols for {region_name}")

    return

In [20]:
# Write out symbol dataframes for each region
output_to_csv(all_data_enriched, output_dir='data/processed')

Finished writing all symbols for Eswatini
Finished writing all symbols for Tanzania
Finished writing all symbols for EAPP
Finished writing all symbols for Malawi
Finished writing all symbols for WAPP
Finished writing all symbols for Serbia
Finished writing all symbols for Timor Leste
Finished writing all symbols for Gambia
Finished writing all symbols for Guinea
Finished writing all symbols for CAPP
Finished writing all symbols for Rwanda
Finished writing all symbols for Turkiye
